# RealPDE Pretraining

Minimal notebook: pull repo, install deps, set config, launch trainer with accelerate, then smoke-test via `local_eval.py`.

**Repo is public** — no `GITHUB_TOKEN` needed. Dataset `realpde` is attached as a Kaggle Dataset (see right panel: `train_sim/train_sim/*.h5`).

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}

In [ ]:
!uv sync

In [ ]:
%pip install -e .

In [ ]:
# --- GPU check ---
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
from pathlib import Path

# =============================================================================
# CONFIG — edit these variables before running
# =============================================================================

# Kaggle dataset paths — auto-detects the nested layout seen in Image 1:
#   /kaggle/input/realpde/train_sim/train_sim/*.h5  (actual on Kaggle)
#   /kaggle/input/realpde/train_sim/*.h5            (flat fallback)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

candidates = [
    '/kaggle/input/realpde/train_sim/train_sim',
    '/kaggle/input/realpde/train_sim',
]
for c in candidates:
    if Path(c).exists() and any(Path(c).glob('*.h5')):
        os.environ['DATA_PATH'] = c
        print(f'DATA_PATH -> {c} ({len(list(Path(c).glob("*.h5")))} .h5 files)')
        break
else:
    # default (will raise a clear error in PDEDataset if still wrong)
    os.environ['DATA_PATH'] = candidates[0]
    print(f'WARNING: no .h5 found in candidates, defaulting to {candidates[0]}')
    for c in candidates:
        print(f'  exists {c}: {Path(c).exists()}')
        if Path(c).exists():
            print(f'    listing: {list(Path(c).glob("*"))[:10]}')
    # also probe one level up for debugging
    _root = Path('/kaggle/input/realpde')
    if _root.exists():
        print(f'  /kaggle/input/realpde contents: {list(_root.iterdir())[:20]}')

# Model
os.environ['MODEL_NAME'] = 'unet'
os.environ['UNET_CHANNELS'] = '16'
os.environ['UNET_N_LAYERS'] = '2'

# Rollout config
os.environ['IN_STEP'] = '20'
os.environ['OUT_STEP'] = '20'
os.environ['INTERVAL'] = '20'

# Training
os.environ['LR'] = '1e-3'
os.environ['EPOCHS'] = '50'
os.environ['BATCH_SIZE'] = '8'

# Paths
os.environ['SAVE_DIR'] = '/kaggle/working/checkpoints'

# Wandb (optional — uses Kaggle Secret WANDB_KEY if set)
try:
    os.environ['WANDB_KEY'] = secrets.get_secret('WANDB_KEY')
except Exception:
    print('WANDB_KEY secret not set — wandb will run in disabled/offline mode if needed.')
os.environ['WANDB_PROJECT'] = 'realpde-pretrain'

print('Config set.')
print(f"  DATA_PATH={os.environ['DATA_PATH']}")

In [ ]:
mixed = 'fp16' if torch.cuda.is_available() else 'no'
!accelerate launch --mixed_precision={mixed} trainer.py

In [ ]:
# --- List saved checkpoints ---
from pathlib import Path
ckpt_dir = Path(os.environ['SAVE_DIR'])
if ckpt_dir.exists():
    for f in sorted(ckpt_dir.glob('*.pth')):
        print(f'{f.name:30s} {f.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('No checkpoints found.')

## Direct eval — how good is your pretrained UNet? (no TTT)

Fast, **non-streaming** check on a hold-out split of `DATA_PATH` (the same 10% `PDEDataset` split the `trainer.py:98` uses). No `local_eval.py` TTT loop — just `MSE` + `rel-L2` in raw space, plus an optional comparison against the **CNO baseline** shipped in `realpde/baseline/` (if attached).

Use `eval_pretrain.py` (repo root) for a one-liner, or the inline block below for plots. Point `--data` elsewhere (e.g. `/kaggle/input/realpde/test_real` or `example_data/test_real`) to score on a different split.


In [ ]:
# --- Direct eval via eval_pretrain.py (one-liner) ---
# Scores /kaggle/working/checkpoints/best.pth on a 10% holdout of DATA_PATH
# and optionally compares to the CNO baseline in /kaggle/input/realpde/baseline/
import shutil, sys
from pathlib import Path

ckpt = Path('/kaggle/working/checkpoints/best.pth')
if not ckpt.exists():
    ckpt = Path('/kaggle/working/checkpoints/final.pth')
if not ckpt.exists():
    print(f'[direct-eval] No checkpoint yet: {ckpt} — train first')
else:
    # auto-find CNO baseline for comparison (if dataset attached)
    baseline_roots = [Path('/kaggle/input/realpde/baseline'), Path('/kaggle/input/realpde/baseline_checkpoints')]
    cno = None
    for root in baseline_roots:
        if root.exists():
            for pat in ['*cno*.pth', '*CNO*.pth']:
                cand = sorted(root.rglob(pat))
                if cand:
                    cand.sort(key=lambda p: (0 if 'sim_real' in p.name else 1, p.name))
                    cno = cand[0]
                    break
        if cno: break
    if cno:
        print(f'[direct-eval] Comparing to CNO baseline: {cno}')
        !python eval_pretrain.py --ckpt {ckpt} --baseline {cno} --baseline-model cno --split val
    else:
        print('[direct-eval] No CNO baseline found — scoring UNet alone')
        print('  (attach the realpde dataset with baseline/ or pass --baseline manually)')
        !python eval_pretrain.py --ckpt {ckpt} --split val
    # Also score on the full split for reference
    print('\n[direct-eval] Full-data reference:')
    !python eval_pretrain.py --ckpt {ckpt} --split full 2>&1 | tail -n 20


In [ ]:
# --- Inline direct eval (no script) — with a quick pred vs target plot ---
import os, torch, matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader, Subset
from datasets import PDEDataset
from models.unet import UNet, UNetConfig

ckpt_path = Path('/kaggle/working/checkpoints/best.pth')
if not ckpt_path.exists():
    ckpt_path = Path('/kaggle/working/checkpoints/final.pth')
if not ckpt_path.exists():
    print(f'No checkpoint at {ckpt_path}')
else:
    ckpt = torch.load(ckpt_path, map_location='cpu')
    cfg = ckpt.get('cfg', {})
    in_step  = int(cfg.get('in_step',  os.environ.get('IN_STEP', 20)))
    out_step = int(cfg.get('out_step', os.environ.get('OUT_STEP', 20)))
    interval = int(cfg.get('interval', os.environ.get('INTERVAL', 20)))
    sub_s    = int(cfg.get('sub_s', 2))
    ch = int(cfg.get('unet_channels', cfg.get('channels', 16)))
    nl = int(cfg.get('unet_n_layers', cfg.get('n_layers', 2)))
    data_path = Path(os.environ.get('DATA_PATH', '/kaggle/input/realpde/train_sim/train_sim'))
    print(f'cfg: in={in_step} out={out_step} interval={interval} ch={ch} nl={nl}')
    print(f'data: {data_path}')
    ds = PDEDataset(data_path, in_step=in_step, out_step=out_step, interval=interval, sub_s=sub_s)
    n_val = max(1, int(0.1*len(ds)))
    val = Subset(ds, range(len(ds)-n_val, len(ds)))
    loader = DataLoader(val, batch_size=8, shuffle=False)
    model = UNet(UNetConfig(in_step=in_step, out_step=out_step, channels=ch, n_layers=nl))
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device).eval()
    import torch.nn.functional as F
    mse, rel, n = 0, 0, 0
    with torch.no_grad():
        for inp, tgt in loader:
            inp, tgt = inp.to(device), tgt.to(device)
            pred = model(inp)
            mse += F.mse_loss(pred, tgt, reduction='sum').item()
            b = pred.shape[0]
            rel += ((pred.reshape(b,-1)-tgt.reshape(b,-1)).norm(dim=1) / tgt.reshape(b,-1).norm(dim=1).clamp_min(1e-8)).sum().item()
            n += b
    mse /= n * in_step*32*64*3
    rel /= n
    print(f'Val (n={n}): MSE {mse:.6f}  rel-L2 {rel:.6f}  (ckpt val_loss {ckpt.get("val_loss","?")})')
    # Plot one sample: input last frame vs pred first frame vs target first frame (u channel)
    inp0, tgt0 = val[0]
    with torch.no_grad():
        pred0 = model(inp0.unsqueeze(0).to(device)).cpu().squeeze(0)
    fig, ax = plt.subplots(1,3, figsize=(12,3))
    v = max(inp0[...,0].abs().max().item(), tgt0[...,0].abs().max().item())
    ax[0].imshow(inp0[-1,...,0], vmin=-v, vmax=v, cmap='RdBu'); ax[0].set_title('input last frame (u)')
    ax[1].imshow(pred0[0,...,0], vmin=-v, vmax=v, cmap='RdBu'); ax[1].set_title('pred first frame (u)')
    ax[2].imshow(tgt0[0,...,0], vmin=-v, vmax=v, cmap='RdBu'); ax[2].set_title('target first frame (u)')
    plt.tight_layout(); plt.show()


## Local eval (streaming TTT smoke test)

Mirrors the official Codabench ingestion loop on the bundled `example_data/` (2 synthetic trajectories, CPU-only) — plus a **CNO baseline** check against the checkpoint shipped in the Kaggle Dataset `realpde/baseline/` (Image 1 right panel).

Three checks:
1. **TinyForecaster smoke test** — `submission.py` as-is (no checkpoint).
2. **CNO baseline** — loads `realpde/baseline/*cno*.pth` via `load_baseline` + `ReferenceTTTModel` and runs `local_eval.py` (the one you asked for).
3. **Trained checkpoint (optional)** — wraps `checkpoints/best.pth` (your UNet) into a temp submission and re-runs the same loop.

All run on CPU via `example_data/` so they work even without the full `test_real` download. Point `--data` at a real `test_real/` + `mean_std_real.pt` folder for leaderboard-comparable numbers (see `local_eval.py --help`).

In [ ]:
# --- 1) TinyForecaster smoke test (no checkpoint) ---
# Uses submission.py / TinyForecaster + example_data/test_real/*.h5
!python local_eval.py --submission . --data ./example_data

In [ ]:
# --- 2) CNO baseline via Kaggle Dataset realpde/baseline/ ---
# Baseline checkpoint lives in the Kaggle Dataset (right panel in Image 1):
#   /kaggle/input/realpde/baseline/  ->  sim_real_cno.pth (or similar)
import shutil, textwrap, sys
from pathlib import Path

# Locate CNO checkpoint inside the Kaggle Dataset (handles flat or nested layout)
baseline_roots = [
    Path('/kaggle/input/realpde/baseline'),
    Path('/kaggle/input/realpde/baseline_checkpoints'),
    Path('/kaggle/input/realpde'),
]
candidates = []
for root in baseline_roots:
    if root.exists():
        candidates += list(root.rglob('*.pth'))
        candidates += list(root.rglob('*.pt'))
print(f'Searching baseline checkpoints: {[str(r) for r in baseline_roots]}')
for c in sorted(set(candidates)):
    try:
        print(f"  {c}  {c.stat().st_size/1024/1024:.1f} MB")
    except Exception:
        print(f"  {c}")

# Prefer a CNO checkpoint (sim_real_cno.pth is the fine-tuned real baseline)
cno_candidates = [p for p in candidates if 'cno' in p.name.lower()]
if cno_candidates:
    cno_candidates.sort(key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
    cno_ckpt = cno_candidates[0]
elif candidates:
    cno_ckpt = sorted(candidates)[0]
    print(f'[warn] No *cno* file found, falling back to {cno_ckpt.name}')
else:
    cno_ckpt = None
    print('[error] No baseline checkpoint found under /kaggle/input/realpde/baseline/')
    print('  Check that the Kaggle Dataset "realpde" is attached and contains baseline/')
    print('  Contents of /kaggle/input/realpde:', list(Path('/kaggle/input/realpde').iterdir()) if Path('/kaggle/input/realpde').exists() else 'missing')

if cno_ckpt is not None:
    print(f'[cno] Selected baseline: {cno_ckpt}')
    eval_dir = Path('/kaggle/working/eval_cno')
    if eval_dir.exists():
        shutil.rmtree(eval_dir)
    eval_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(cno_ckpt, eval_dir / 'model.pth')
    print(f'[cno] Copied {cno_ckpt.name} -> {eval_dir}/model.pth')
    (eval_dir / 'submission.py').write_text(textwrap.dedent(f'''
        """CNO baseline submission for local_eval (auto-generated)."""
        import sys
        from pathlib import Path
        REPO_DIR = "{Path('/kaggle/working/realpde').as_posix()}"
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        import torch
        from load_baseline import load_baseline
        from submission_template import ReferenceTTTModel
        BASELINE_CKPT = "{cno_ckpt.as_posix()}"
        def get_ttt_model(submission_dir, device):
            ckpt = str(Path(submission_dir) / "model.pth")
            import os
            if not Path(ckpt).exists():
                ckpt = BASELINE_CKPT
            print(f"[cno submission] loading {{ckpt}} on {{device}}")
            base, meta = load_baseline("cno", ckpt, device=device)
            print(f"[cno submission] loaded {{meta}}")
            return ReferenceTTTModel(base, device)
    '''))
    print(f'[cno] Wrote {eval_dir}/submission.py')
    REPO_DIR = '/kaggle/working/realpde'
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    import subprocess
    print('[cno] Running local_eval on example_data (CPU)...')
    result = subprocess.run([sys.executable, 'local_eval.py', '--submission', str(eval_dir), '--data', './example_data'])
    print(f'[cno] local_eval exit code: {result.returncode}')
    real_test = Path('/kaggle/input/realpde/test_real')
    real_data_root = Path('/kaggle/input/realpde')
    if real_test.exists() and (real_data_root / 'mean_std_real.pt').exists():
        print(f'[cno] Real test set found at {real_test} — scoring on it too')
        result2 = subprocess.run([sys.executable, 'local_eval.py', '--submission', str(eval_dir), '--data', str(real_data_root)])
        print(f'[cno] local_eval (real data) exit code: {result2.returncode}')
    else:
        print('[cno] No real test_real/ + mean_std_real.pt under /kaggle/input/realpde — skipping real-data score')
        print('      (attach the full test split or point --data at your download to get leaderboard-comparable numbers)')


In [ ]:
# --- 3) Trained checkpoint smoke test (optional) ---
# Packs /kaggle/working/checkpoints/best.pth into a temp submission that
# implements get_ttt_model() via your UNet, then runs local_eval on it.
import shutil, textwrap
from pathlib import Path

ckpt = Path(os.environ.get('SAVE_DIR', '/kaggle/working/checkpoints')) / 'best.pth'
eval_dir = Path('/kaggle/working/eval_submission')

if not ckpt.exists():
    print(f'[skip] No checkpoint at {ckpt} — run training first or check SAVE_DIR.')
else:
    eval_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(ckpt, eval_dir / 'model.pth')
    if Path('ttt_model.py').exists():
        shutil.copy('ttt_model.py', eval_dir / 'ttt_model.py')
    (eval_dir / 'submission.py').write_text(textwrap.dedent('''
        import os
        from pathlib import Path
        import torch
        try:
            from ttt_model import TTTModel
        except Exception:
            import torch.nn as nn; TTTModel = nn.Module
        from models.unet import UNet, UNetConfig
        from submission import ReferenceTTTModel
        def get_ttt_model(submission_dir, device):
            in_step  = int(os.environ.get("IN_STEP", 20))
            out_step = int(os.environ.get("OUT_STEP", 20))
            channels = int(os.environ.get("UNET_CHANNELS", 16))
            n_layers = int(os.environ.get("UNET_N_LAYERS", 2))
            cfg = UNetConfig(in_step=in_step, out_step=out_step, channels=channels, n_layers=n_layers)
            base = UNet(cfg)
            ckpt = Path(submission_dir) / "model.pth"
            state = torch.load(ckpt, map_location="cpu")
            if isinstance(state, dict) and "model_state_dict" in state:
                state = state["model_state_dict"]
            base.load_state_dict(state)
            return ReferenceTTTModel(base, device)
    '''))
    print(f'[eval] Packed {ckpt} -> {eval_dir}/model.pth')
    print(f'[eval] submission files: {list(eval_dir.iterdir())}')
    import sys; REPO_DIR not in sys.path and sys.path.insert(0, REPO_DIR)
    import subprocess, sys as _sys
    result = subprocess.run([_sys.executable, 'local_eval.py', '--submission', str(eval_dir), '--data', './example_data'])
    print(f'[eval] exit code: {result.returncode}')
